# Hybrid Flood and Cyclone Risk Prediction System  
### Using HistGradientBoosting & Automatic Severity Scoring

This notebook implements a hybrid machine learning framework to:
- Classify **Flood** and **Cyclone** risk states
- Automatically compute **severity scores**
- Generate **hybrid risk indices** combining probability and severity


# Data handling

In [1]:

import pandas as pd
import numpy as np

# Encoding


In [2]:
from sklearn.preprocessing import LabelEncoder

# Machine Learning Models


In [3]:
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor
)

### Basic Numeric Cleaning Functions

In [4]:
def basic_numeric_clean(df):
    """
    Replace infinite values with NaN 
    to avoid model instability.
    """
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
    return df


### Supporting Utility Functions

In [5]:
def safe_pressure_anomaly(mslp, low_thresh):
    """
    Normalized pressure anomaly calculation
    for cyclone severity estimation.
    """
    return (low_thresh - mslp) / abs(low_thresh)


def iqr_threshold(series):
    """
    Compute upper and lower thresholds
    using Interquartile Range (IQR).
    """
    Q1, Q3 = series.quantile([0.25, 0.75])
    IQR = Q3 - Q1
    return Q3 + 1.5 * IQR, Q1 - 1.5 * IQR


### Automatic Feature Weight Computation

In [6]:
def compute_auto_weights(X, y):
    """
    Automatically assign feature weights
    using variance-based proxy importance.
    """
    model = HistGradientBoostingRegressor(
        max_depth=3,
        learning_rate=0.05,
        max_iter=200,
        random_state=42
    )
    model.fit(X, y)
    
    # Variance-based weighting
    var = X.var().values
    weights = var / var.sum()
    return weights


# Load Dataset

In [7]:
df = pd.read_excel("Balood_data.xlsx")

# Preview data
df.head()


,INDEX,YEAR,MN,.MMAX,.HMAX,.MMIN,.LMIN,..TMRF,HVYRF,RD,...,SW,.W,NW,CA,VA,.AWS,Cyclone,Flood,Flood_Severity,Cyclone_Severity
0,42948,1980,12,27.4,29.1,13.7,10.1,6.4,5,1,...,0,0,0,11,0,1.0,NaN,NaN,NaN,NaN
1,42948,1981,1,26,31.8,12.6,7.4,16.8,10.4,2,...,0,0,0,22,0,4.0,NaN,NaN,NaN,NaN
2,42948,1981,2,32.3,36.1,15.6,11.1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,42948,1981,3,34.6,38.3,18.6,14.9,98,33.2,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,42948,1981,4,40,43.6,24.1,18.4,0,0,0,...,0,0,0,30,0,NaN,NaN,NaN,NaN,NaN


### Clean Column Names

In [8]:
# Standardize column names
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.replace('.', '', regex=False)
    .str.replace(' ', '_')
)

### Encode Categorical Variables

In [9]:
# Label encoding for categorical columns
for col in df.select_dtypes(include="object").columns:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

### Basic Data Cleaning

In [10]:
# Handle NaN and infinite values
df = basic_numeric_clean(df)


### Compute Statistical Thresholds

In [11]:
thresh_TOTRF_high, _ = iqr_threshold(df["TOTRF"])
thresh_RD_high, _    = iqr_threshold(df["RD"])
thresh_RH_high, _    = iqr_threshold(df["RH"])
thresh_DBT_high, _   = iqr_threshold(df["DBT"])
thresh_MWS_high, _   = iqr_threshold(df["MWS"])
_, thresh_MSLP_low   = iqr_threshold(df["MSLP"])


### Flood & Cyclone State Classification

In [12]:
mean_TOTRF = df["TOTRF"].mean()
mean_MWS   = df["MWS"].mean()

# Flood State
df["Flood_State"] = np.select(
    [
        df["TOTRF"] <= mean_TOTRF,
        (df["TOTRF"] > mean_TOTRF) & (df["TOTRF"] <= thresh_TOTRF_high),
        df["TOTRF"] > thresh_TOTRF_high
    ],
    [0, 1, 2]
)

# Cyclone State
df["Cyclone_State"] = np.select(
    [
        df["MWS"] <= mean_MWS,
        (df["MWS"] > mean_MWS) & (df["MWS"] <= thresh_MWS_high),
        df["MWS"] > thresh_MWS_high
    ],
    [0, 1, 2]
)


### State Distribution Analysis

In [13]:
print("Flood State Distribution:")
print(df["Flood_State"].value_counts())

print("\nCyclone State Distribution:")
print(df["Cyclone_State"].value_counts())


Flood State Distribution:
1    324
0    207
Name: Flood_State, dtype: int64

Cyclone State Distribution:
1    268
0    263
Name: Cyclone_State, dtype: int64


### Flood Severity Score Calculation

In [14]:
flood_features = pd.DataFrame({
    "TOTRF": df["TOTRF"] / thresh_TOTRF_high,
    "RD": df["RD"] / thresh_RD_high,
    "RH": df["RH"] / thresh_RH_high,
    "DBT": df["DBT"] / thresh_DBT_high
})

flood_weights = compute_auto_weights(
    flood_features, df["Flood_State"]
)

df["Flood_Severity"] = flood_features.values @ flood_weights


### Cyclone Severity Score Calculation

In [15]:
cyclone_features = pd.DataFrame({
    "MWS": df["MWS"] / thresh_MWS_high,
    "MSLP": safe_pressure_anomaly(df["MSLP"], thresh_MSLP_low),
    "RH": df["RH"] / thresh_RH_high,
    "DBT": df["DBT"] / thresh_DBT_high
})

cyclone_weights = compute_auto_weights(
    cyclone_features, df["Cyclone_State"]
)

df["Cyclone_Severity"] = cyclone_features.values @ cyclone_weights


### Feature Selection

In [16]:
FEATURES = df.drop(columns=[
    "Flood_State",
    "Cyclone_State",
    "Flood_Severity",
    "Cyclone_Severity"
])


### Model Training (HistGradientBoosting)

In [17]:
flood_clf = HistGradientBoostingClassifier(
    max_depth=3,
    learning_rate=0.05,
    max_iter=200,
    random_state=42
)

cyclone_clf = HistGradientBoostingClassifier(
    max_depth=3,
    learning_rate=0.05,
    max_iter=200,
    random_state=42
)

flood_clf.fit(FEATURES, df["Flood_State"])
cyclone_clf.fit(FEATURES, df["Cyclone_State"])


HistGradientBoostingClassifier(learning_rate=0.05, max_depth=3, max_iter=200,
                               random_state=42)

### Final Hybrid Risk Output

In [18]:
sample = FEATURES.head(5)

final_predictions = pd.DataFrame({
    "Flood_Probability": flood_clf.predict_proba(sample)[:, 1],
    "Flood_Severity": df.loc[sample.index, "Flood_Severity"],
    "Cyclone_Probability": cyclone_clf.predict_proba(sample)[:, 1],
    "Cyclone_Severity": df.loc[sample.index, "Cyclone_Severity"]
})

final_predictions["Flood_Hybrid_Risk"] = (
    final_predictions["Flood_Probability"] *
    final_predictions["Flood_Severity"]
)

final_predictions["Cyclone_Hybrid_Risk"] = (
    final_predictions["Cyclone_Probability"] *
    final_predictions["Cyclone_Severity"]
)

final_predictions


,Flood_Probability,Flood_Severity,Cyclone_Probability,Cyclone_Severity,Flood_Hybrid_Risk,Cyclone_Hybrid_Risk
0,0.999983,0.211488,0.000023,-5.970071,0.211484,-0.000134
1,0.000027,0.185252,0.000023,-5.971880,0.000005,-0.000134
2,0.000027,0.183110,0.999978,-5.965486,0.000005,-5.965354
3,0.000027,0.211984,0.999978,-5.966282,0.000006,-5.966150
4,0.999983,0.231659,0.000023,-5.972717,0.231655,-0.000134


### Display Feature Weights

In [20]:
print("Flood Severity Weights:")
print(dict(zip(flood_features.columns, flood_weights)))

print("\nCyclone Severity Weights:")
print(dict(zip(cyclone_features.columns, cyclone_weights)))


Flood Severity Weights:
{'TOTRF': 0.2669587970476118, 'RD': 0.2620042469351464, 'RH': 0.23631948572552897, 'DBT': 0.2347174702917128}

Cyclone Severity Weights:
{'MWS': 0.013902018074619943, 'MSLP': 0.9647483836711591, 'RH': 0.010711104543779068, 'DBT': 0.010638493710441854}
